In [ ]:
import os
import json
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

### Dataset

In [ ]:
class MultiLabelDataset(Dataset):
    """
    Reads from the JSON split files produced by preprocessing.py.
    Each entry has:
        { "image": "<path>", "labels": [0, 1, 0, 1, 0] }
    """
    def __init__(self, json_file, transform=None):
        with open(json_file) as f:
            self.data = json.load(f)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(item["labels"], dtype=torch.float32)
        return image, labels

### Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

### Loading Datasets

In [ ]:
PROCESSED_DIR = "processed_dataset"
NUM_CLASSES = 5

train_dataset = MultiLabelDataset(
    os.path.join(PROCESSED_DIR, "train.json"),
    transform=train_transform
)

val_dataset = MultiLabelDataset(
    os.path.join(PROCESSED_DIR, "val.json"),
    transform=val_transform
)

test_dataset = MultiLabelDataset(
    os.path.join(PROCESSED_DIR, "test.json"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

### Evaluation Helpers

In [ ]:
def evaluate(model, loader, tag=""):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.numpy())

    all_probs  = np.vstack(all_probs)
    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall    = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1_per    = f1_score(all_labels, all_preds, average=None, zero_division=0)
    f1_macro  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_micro  = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    auc       = roc_auc_score(all_labels, all_probs, average=None)

    print(f"\n=== {tag} ===")
    print(f"{'Class':<8} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}")
    for i in range(NUM_CLASSES):
        print(f"  [{i}]   {precision[i]:>10.4f} {recall[i]:>8.4f} {f1_per[i]:>8.4f} {auc[i]:>8.4f}")
    print(f"Macro F1 : {f1_macro:.4f}")
    print(f"Micro F1 : {f1_micro:.4f}")

    return all_probs, all_preds, all_labels


def plot_roc_curves(all_labels, all_probs, tag=""):
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 4))
    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(all_labels[:, i], all_probs[:, i])
        auc_val = roc_auc_score(all_labels[:, i], all_probs[:, i])
        axes[i].plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
        axes[i].plot([0, 1], [0, 1], "k--")
        axes[i].set_title(f"Class {i}")
        axes[i].set_xlabel("FPR")
        axes[i].set_ylabel("TPR")
        axes[i].legend()
    fig.suptitle(f"ROC Curves — {tag}")
    plt.tight_layout()
    plt.savefig(f"roc_{tag.lower().replace(' ', '_')}.png", dpi=120)
    plt.show()


def tune_thresholds(all_labels, all_probs):
    thresholds = np.arange(0.1, 0.91, 0.05)
    best_thresholds, best_f1s = [], []
    for i in range(NUM_CLASSES):
        best_f1, best_t = 0, 0.5
        for t in thresholds:
            preds = (all_probs[:, i] > t).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
        best_f1s.append(best_f1)
    print("Best thresholds per class:", best_thresholds)
    print("Best F1 per class:        ", best_f1s)
    return best_thresholds


def evaluate_tuned(all_labels, all_probs, best_thresholds, tag=""):
    tuned_preds = np.zeros_like(all_probs, dtype=int)
    for i in range(NUM_CLASSES):
        tuned_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)
    f1_macro = f1_score(all_labels, tuned_preds, average="macro", zero_division=0)
    f1_micro = f1_score(all_labels, tuned_preds, average="micro", zero_division=0)
    print(f"\n=== {tag} — After Threshold Tuning ===")
    print(f"Precision : {precision_score(all_labels, tuned_preds, average=None, zero_division=0)}")
    print(f"Recall    : {recall_score(all_labels, tuned_preds, average=None, zero_division=0)}")
    print(f"Macro F1  : {f1_macro:.4f}")
    print(f"Micro F1  : {f1_micro:.4f}")

### Strategy 1 — Training from Scratch (ResNet50)

In [ ]:
scratch_model = models.resnet50(weights=None)
scratch_model.fc = nn.Linear(scratch_model.fc.in_features, NUM_CLASSES)
scratch_model = scratch_model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

In [ ]:
epochs = 10

for epoch in range(epochs):
    scratch_model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = scratch_model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()
    print(f"[Scratch] Epoch {epoch+1}/{epochs}  Loss: {running_loss:.4f}  LR: {scheduler.get_last_lr()[0]:.2e}")

In [ ]:
scratch_probs, scratch_preds, scratch_labels = evaluate(scratch_model, val_loader, tag="Scratch — Validation")
plot_roc_curves(scratch_labels, scratch_probs, tag="Scratch Validation")

In [ ]:
scratch_thresholds = tune_thresholds(scratch_labels, scratch_probs)
evaluate_tuned(scratch_labels, scratch_probs, scratch_thresholds, tag="Scratch")

In [ ]:
torch.save(scratch_model.state_dict(), "resnet50_scratch.pth")
print("Saved: resnet50_scratch.pth")

### Strategy 2 — Transfer Learning (ResNet50, ImageNet weights)

In [ ]:
transfer_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all backbone layers
for param in transfer_model.parameters():
    param.requires_grad = False

# Replace and unfreeze the final FC layer
transfer_model.fc = nn.Linear(transfer_model.fc.in_features, NUM_CLASSES)
for param in transfer_model.fc.parameters():
    param.requires_grad = True

transfer_model = transfer_model.to(device)

criterion_tl = nn.BCEWithLogitsLoss()
optimizer_tl = torch.optim.Adam(transfer_model.fc.parameters(), lr=1e-3)
scheduler_tl = torch.optim.lr_scheduler.StepLR(optimizer_tl, step_size=3, gamma=0.5)

In [ ]:
# Phase 1 — train head only
num_epochs_phase1 = 5

for epoch in range(num_epochs_phase1):
    transfer_model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = transfer_model(images)
        loss = criterion_tl(outputs, labels)
        optimizer_tl.zero_grad()
        loss.backward()
        optimizer_tl.step()
        running_loss += loss.item()

    scheduler_tl.step()
    print(f"[Transfer Phase-1] Epoch {epoch+1}/{num_epochs_phase1}  Loss: {running_loss:.4f}")

# Phase 2 — unfreeze layer3 and layer4 for fine-tuning
print("\nUnfreezing layer3 and layer4 for fine-tuning...")
for name, param in transfer_model.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True

optimizer_ft = torch.optim.Adam(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=1e-4
)
scheduler_ft = torch.optim.lr_scheduler.StepLR(optimizer_ft, step_size=3, gamma=0.5)

num_epochs_phase2 = 5

for epoch in range(num_epochs_phase2):
    transfer_model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = transfer_model(images)
        loss = criterion_tl(outputs, labels)
        optimizer_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()
        running_loss += loss.item()

    scheduler_ft.step()
    print(f"[Transfer Phase-2] Epoch {epoch+1}/{num_epochs_phase2}  Loss: {running_loss:.4f}")

In [ ]:
transfer_probs, transfer_preds, transfer_labels = evaluate(transfer_model, val_loader, tag="Transfer — Validation")
plot_roc_curves(transfer_labels, transfer_probs, tag="Transfer Validation")

In [ ]:
transfer_thresholds = tune_thresholds(transfer_labels, transfer_probs)
evaluate_tuned(transfer_labels, transfer_probs, transfer_thresholds, tag="Transfer")

In [ ]:
torch.save(transfer_model.state_dict(), "resnet50_transfer.pth")
print("Saved: resnet50_transfer.pth")

### Final Test Set Evaluation

In [ ]:
# Evaluate best model (transfer) on held-out test set
test_probs, test_preds, test_labels = evaluate(transfer_model, test_loader, tag="Transfer — Test Set")
plot_roc_curves(test_labels, test_probs, tag="Transfer Test")
evaluate_tuned(test_labels, test_probs, transfer_thresholds, tag="Transfer Test")